# 2. Train Transformer Model (Google Colab)

**Chiến lược:**
- Mỗi epoch train trên **subset ngẫu nhiên** (~500k câu) → epoch ngắn → checkpoint thường xuyên.
- **LR Scheduler:** Adam(3e-4) + ReduceLROnPlateau — đơn giản, ổn định, không bị near-zero LR.
- **Auto-Resume:** Tự động tiếp tục nếu Colab bị ngắt.
- **Biểu đồ:** Vẽ 1 lần sau khi train xong.

> ⚠️ **Lần đầu chạy:** Hãy đảm bảo xóa file `transformer_latest.pt` cũ (nếu có) trên Drive để tránh load checkpoint với LR sai.

In [ ]:
!pip install tokenizers torch matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import math, os, json, random
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from tokenizers import Tokenizer
from torch.cuda.amp import GradScaler, autocast

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Multilingual_MT'
    print('✅ Google Colab + Drive đã kết nối.')
except:
    BASE_DIR = '/kaggle/input/multilingual-mt-data'
    print('⚠️ Kaggle mode.')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## ⚙️ Siêu tham số

In [ ]:
# ============================================================
#  SIÊU THAM SỐ
# ============================================================
EPOCHS       = 50           # Chạy nhiều epoch hơn vì mỗi epoch ngắn
SUBSET_SIZE  = 500_000      # Câu mỗi epoch
BATCH_SIZE   = 64
ACCUM_STEPS  = 4            # Effective batch = 256
LR           = 3e-4         # ✅ LR cố định ban đầu, scheduler tự giảm
MAX_LEN      = 128

OUT_DIR = f"{BASE_DIR}/model_assets"
os.makedirs(OUT_DIR, exist_ok=True)

steps_per_epoch = (SUBSET_SIZE // BATCH_SIZE) // ACCUM_STEPS
print(f'Steps/epoch (ước lượng): {steps_per_epoch:,}')
print(f'Tổng steps ({EPOCHS} epochs): {steps_per_epoch * EPOCHS:,}')

## Kiến trúc Transformer-Base

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return self.dropout(x + self.pe.transpose(0, 1)[:, :x.size(1), :])

class TransformerMT(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8,
                 num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.1, pad_idx=0):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_enc   = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, vocab_size)

    def create_mask(self, src, tgt):
        T = tgt.shape[1]
        tgt_mask = torch.triu(torch.ones(T, T, device=src.device), diagonal=1).bool()
        tgt_mask = tgt_mask.float().masked_fill(tgt_mask, float('-inf'))
        src_mask = torch.zeros(src.shape[1], src.shape[1], device=src.device).bool()
        return src_mask, tgt_mask, (src == self.pad_idx), (tgt == self.pad_idx)

    def forward(self, src, tgt):
        sm, tm, sp, tp = self.create_mask(src, tgt)
        se = self.pos_enc(self.embedding(src) * math.sqrt(self.d_model))
        te = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))
        out = self.transformer(se, te, src_mask=sm, tgt_mask=tm,
                               src_key_padding_mask=sp, tgt_key_padding_mask=tp,
                               memory_key_padding_mask=sp)
        return self.fc_out(out)

## Dataset

In [ ]:
tokenizer = Tokenizer.from_file(f"{BASE_DIR}/tokenizer/tokenizer.json")
PAD_IDX = tokenizer.token_to_id('[PAD]')
BOS_IDX = tokenizer.token_to_id('[BOS]')
EOS_IDX = tokenizer.token_to_id('[EOS]')
print(f'Vocab size: {tokenizer.get_vocab_size():,}')

class TranslationDataset(Dataset):
    def __init__(self, path):
        print(f'Đang đọc {path} ...')
        with open(path, 'r', encoding='utf-8') as f:
            self.lines = f.readlines()
        print(f'✅ Tổng: {len(self.lines):,} câu')

    def __len__(self): return len(self.lines)

    def __getitem__(self, idx):
        parts = self.lines[idx].strip().split('\t')
        if len(parts) != 3:
            return torch.tensor([BOS_IDX, EOS_IDX]), torch.tensor([BOS_IDX, EOS_IDX])
        tag, src, tgt = parts
        s = tokenizer.encode(f'{tag} {src}').ids[:MAX_LEN - 2]
        t = tokenizer.encode(tgt).ids[:MAX_LEN - 2]
        return (torch.tensor([BOS_IDX] + s + [EOS_IDX], dtype=torch.long),
                torch.tensor([BOS_IDX] + t + [EOS_IDX], dtype=torch.long))

def collate_fn(batch):
    src, tgt = zip(*batch)
    return (pad_sequence(src, padding_value=PAD_IDX, batch_first=True),
            pad_sequence(tgt, padding_value=PAD_IDX, batch_first=True))

full_dataset = TranslationDataset(f"{BASE_DIR}/data/processed/train.txt")
TOTAL_SIZE   = len(full_dataset)

## 🚀 Training Loop

In [ ]:
model     = TransformerMT(vocab_size=tokenizer.get_vocab_size(), pad_idx=PAD_IDX).to(DEVICE)

# ✅ Option A: Adam cố định + ReduceLROnPlateau (đơn giản, ổn định)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, min_lr=1e-6, verbose=True
)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
scaler    = GradScaler()

history     = {'epoch': [], 'train_loss': [], 'lr': []}
start_epoch = 1
best_loss   = float('inf')

# --- Auto-Resume (chỉ dùng nếu đây là checkpoint MỚI với LR đúng) ---
RESUME_CKPT = f"{OUT_DIR}/transformer_v2_latest.pt"  # Tên mới để tránh load checkpoint cũ bị sai LR
if os.path.exists(RESUME_CKPT):
    print('🔄 Tìm thấy checkpoint v2, tiếp tục...')
    ckpt = torch.load(RESUME_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    history     = ckpt.get('history', history)
    start_epoch = ckpt['epoch'] + 1
    best_loss   = ckpt.get('best_loss', float('inf'))
    print(f'✅ Resume từ Epoch {ckpt["epoch"]} | Best loss: {best_loss:.4f}')
else:
    print('🆕 Bắt đầu training mới (v2 — LR đã sửa).')

# =================== TRAINING LOOP ===================
print(f'\nBắt đầu | {start_epoch} → {EPOCHS} epochs | {SUBSET_SIZE:,} câu/epoch | LR={LR}\n')

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()

    # Subset ngẫu nhiên
    indices = random.sample(range(TOTAL_SIZE), min(SUBSET_SIZE, TOTAL_SIZE))
    loader  = DataLoader(Subset(full_dataset, indices),
                         batch_size=BATCH_SIZE, shuffle=True,
                         collate_fn=collate_fn, num_workers=2, pin_memory=True)

    total_loss = 0
    optimizer.zero_grad()

    for i, (src, tgt) in enumerate(loader):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        with autocast():
            logits = model(src, tgt[:, :-1])
            loss   = criterion(logits.reshape(-1, logits.shape[-1]),
                               tgt[:, 1:].reshape(-1)) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS

        if i % 500 == 0:
            cur_lr = optimizer.param_groups[0]['lr']
            print(f'  Ep {epoch:02d}/{EPOCHS} | Step {i:,}/{len(loader):,} | '
                  f'Loss: {loss.item()*ACCUM_STEPS:.4f} | LR: {cur_lr:.2e}')

    avg_loss = total_loss / len(loader)
    cur_lr   = optimizer.param_groups[0]['lr']

    # ReduceLROnPlateau giảm LR nếu loss không giảm sau N epoch
    scheduler.step(avg_loss)

    history['epoch'].append(epoch)
    history['train_loss'].append(avg_loss)
    history['lr'].append(cur_lr)

    is_best = avg_loss < best_loss
    if is_best:
        best_loss = avg_loss

    print(f'\n✅ Ep {epoch:02d}/{EPOCHS} | Loss: {avg_loss:.4f} | '
          f'Best: {best_loss:.4f} | LR: {cur_lr:.2e}'
          + (' ⭐ NEW BEST' if is_best else ''))

    # Lưu checkpoint
    save_data = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'loss': avg_loss,
        'best_loss': best_loss,
        'history': history
    }
    torch.save(save_data, f"{OUT_DIR}/transformer_v2_latest.pt")
    torch.save(save_data, f"{OUT_DIR}/transformer_v2_ep{epoch:02d}.pt")
    if is_best:
        torch.save(save_data, f"{OUT_DIR}/transformer_v2_best.pt")  # Model tốt nhất
        print('🏆 Đã lưu BEST model!')

    with open(f"{OUT_DIR}/transformer_v2_history.json", 'w') as f:
        json.dump(history, f)
    print(f'💾 Checkpoint saved.\n' + '-'*60)

print('\n🎉 TRAINING HOÀN TẤT!')

## 📊 Biểu đồ kết quả (chạy sau khi train xong)

In [ ]:
import json, matplotlib.pyplot as plt

with open(f"{OUT_DIR}/transformer_v2_history.json") as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('Transformer-Base v2 — Training Summary', color='white', fontsize=16, y=1.02)

for ax, key, color, title, ylabel in [
    (axes[0], 'train_loss', '#6366f1', 'Training Loss', 'Loss'),
    (axes[1], 'lr',         '#10b981', 'Learning Rate', 'LR'),
]:
    ax.set_facecolor('#1a1d2e')
    ax.plot(history['epoch'], history[key], color=color,
            linewidth=2.5, marker='o', markersize=6)
    ax.fill_between(history['epoch'], history[key], alpha=0.15, color=color)
    ax.set_title(title, color='white', fontsize=14)
    ax.set_xlabel('Epoch', color='#94a3b8')
    ax.set_ylabel(ylabel, color='#94a3b8')
    ax.tick_params(colors='#94a3b8')
    ax.spines[:].set_color('#2d3748')
    ax.grid(True, alpha=0.2, color='#4a5568', linestyle='--')

    min_val = min(history[key])
    min_ep  = history['epoch'][history[key].index(min_val)]
    ax.annotate(f'min={min_val:.4f}', xy=(min_ep, min_val),
                xytext=(min_ep + 0.5, min_val * 1.05),
                color=color, fontsize=9,
                arrowprops=dict(arrowstyle='->', color=color))

plt.tight_layout()
out_img = f"{OUT_DIR}/transformer_v2_curve.png"
plt.savefig(out_img, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'✅ Saved: {out_img}')